# Create separate Campaign / Adset / Ad performance views

Source of truth: `Gold.rpt_unified_ad_performance` (ad × day, Meta + Google).

Creates:
- `Gold.vw_ad_performance` — ad × day (alias of unified)
- `Gold.vw_adset_performance` — adset × day rollup
- `Gold.vw_campaign_performance` — campaign × day rollup



In [ ]:
S = "Gold"
SRC = f"{S}.rpt_unified_ad_performance"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {S}")
print("source rows", spark.table(SRC).count())
print("by platform")
spark.table(SRC).groupBy("platform").count().show()



In [ ]:
# AD view — native grain (ad × day), Meta + Google
spark.sql(f'''
CREATE OR REPLACE VIEW {S}.vw_ad_performance AS
SELECT
  platform,
  full_date, year, month, month_name, day_name,
  tenant_id, connector_id,
  account_id, account_name,
  campaign_id, campaign_name, campaign_status, campaign_channel_or_objective,
  daily_budget_inr, campaign_daily_budget_inr,
  adset_id, adset_name, adset_status,
  optimization_goal, billing_event,
  age_min, age_max, age_range, geo_country, geo_regions, geo_cities,
  ad_id, ad_name, ad_type, ad_status, creative_id, headline, description, final_urls,
  impressions, reach, frequency, clicks, unique_clicks, inline_link_clicks, unique_ctr,
  ctr, cpm, cpp, cpc, spend, spend_inr,
  leads, cost_per_lead, link_clicks, landing_page_views, post_engagement, video_views_3s,
  conversions, conversions_value, cost_per_conversion, roas, engagements, video_views,
  gold_processed_at
FROM {SRC}
''')
print("[OK] Gold.vw_ad_performance")



In [ ]:
# ADSET view — rollup ad → adset × day (Meta + Google)
spark.sql(f'''
CREATE OR REPLACE VIEW {S}.vw_adset_performance AS
SELECT
  platform,
  full_date, year, month, month_name, day_name,
  MAX(tenant_id) AS tenant_id,
  MAX(connector_id) AS connector_id,
  account_id, MAX(account_name) AS account_name,
  campaign_id, MAX(campaign_name) AS campaign_name,
  MAX(campaign_status) AS campaign_status,
  MAX(campaign_channel_or_objective) AS campaign_channel_or_objective,
  MAX(daily_budget_inr) AS daily_budget_inr,
  adset_id, MAX(adset_name) AS adset_name, MAX(adset_status) AS adset_status,
  MAX(optimization_goal) AS optimization_goal,
  MAX(billing_event) AS billing_event,
  MAX(age_min) AS age_min, MAX(age_max) AS age_max, MAX(age_range) AS age_range,
  MAX(geo_country) AS geo_country, MAX(geo_regions) AS geo_regions, MAX(geo_cities) AS geo_cities,
  SUM(impressions) AS impressions,
  SUM(reach) AS reach,
  CASE WHEN SUM(reach) > 0 THEN SUM(impressions) / SUM(reach) ELSE NULL END AS frequency,
  SUM(clicks) AS clicks,
  SUM(unique_clicks) AS unique_clicks,
  SUM(inline_link_clicks) AS inline_link_clicks,
  CASE WHEN SUM(impressions) > 0 THEN SUM(clicks) / SUM(impressions) ELSE NULL END AS ctr,
  CASE WHEN SUM(impressions) > 0 THEN (SUM(spend) / SUM(impressions)) * 1000 ELSE NULL END AS cpm,
  CASE WHEN SUM(reach) > 0 THEN (SUM(spend) / SUM(reach)) * 1000 ELSE NULL END AS cpp,
  CASE WHEN SUM(clicks) > 0 THEN SUM(spend) / SUM(clicks) ELSE NULL END AS cpc,
  SUM(spend) AS spend,
  SUM(spend_inr) AS spend_inr,
  SUM(leads) AS leads,
  CASE WHEN SUM(leads) > 0 THEN SUM(spend) / SUM(leads) ELSE NULL END AS cost_per_lead,
  SUM(link_clicks) AS link_clicks,
  SUM(landing_page_views) AS landing_page_views,
  SUM(post_engagement) AS post_engagement,
  SUM(video_views_3s) AS video_views_3s,
  SUM(conversions) AS conversions,
  SUM(conversions_value) AS conversions_value,
  CASE WHEN SUM(conversions) > 0 THEN SUM(spend) / SUM(conversions) ELSE NULL END AS cost_per_conversion,
  CASE WHEN SUM(spend) > 0 THEN SUM(conversions_value) / SUM(spend) ELSE NULL END AS roas,
  SUM(engagements) AS engagements,
  SUM(video_views) AS video_views,
  COUNT(DISTINCT ad_id) AS ad_count,
  MAX(gold_processed_at) AS gold_processed_at
FROM {SRC}
GROUP BY
  platform, full_date, year, month, month_name, day_name,
  account_id, campaign_id, adset_id
''')
print("[OK] Gold.vw_adset_performance")



In [ ]:
# CAMPAIGN view — rollup ad → campaign × day (Meta + Google)
spark.sql(f'''
CREATE OR REPLACE VIEW {S}.vw_campaign_performance AS
SELECT
  platform,
  full_date, year, month, month_name, day_name,
  MAX(tenant_id) AS tenant_id,
  MAX(connector_id) AS connector_id,
  account_id, MAX(account_name) AS account_name,
  campaign_id, MAX(campaign_name) AS campaign_name,
  MAX(campaign_status) AS campaign_status,
  MAX(campaign_channel_or_objective) AS campaign_channel_or_objective,
  MAX(daily_budget_inr) AS daily_budget_inr,
  MAX(campaign_daily_budget_inr) AS campaign_daily_budget_inr,
  MAX(buying_type) AS buying_type,
  MAX(campaign_bid_strategy) AS campaign_bid_strategy,
  MAX(budget_remaining) AS budget_remaining,
  SUM(impressions) AS impressions,
  SUM(reach) AS reach,
  CASE WHEN SUM(reach) > 0 THEN SUM(impressions) / SUM(reach) ELSE NULL END AS frequency,
  SUM(clicks) AS clicks,
  SUM(unique_clicks) AS unique_clicks,
  SUM(inline_link_clicks) AS inline_link_clicks,
  CASE WHEN SUM(impressions) > 0 THEN SUM(clicks) / SUM(impressions) ELSE NULL END AS ctr,
  CASE WHEN SUM(impressions) > 0 THEN (SUM(spend) / SUM(impressions)) * 1000 ELSE NULL END AS cpm,
  CASE WHEN SUM(reach) > 0 THEN (SUM(spend) / SUM(reach)) * 1000 ELSE NULL END AS cpp,
  CASE WHEN SUM(clicks) > 0 THEN SUM(spend) / SUM(clicks) ELSE NULL END AS cpc,
  SUM(spend) AS spend,
  SUM(spend_inr) AS spend_inr,
  SUM(leads) AS leads,
  CASE WHEN SUM(leads) > 0 THEN SUM(spend) / SUM(leads) ELSE NULL END AS cost_per_lead,
  SUM(link_clicks) AS link_clicks,
  SUM(landing_page_views) AS landing_page_views,
  SUM(post_engagement) AS post_engagement,
  SUM(video_views_3s) AS video_views_3s,
  SUM(conversions) AS conversions,
  SUM(conversions_value) AS conversions_value,
  CASE WHEN SUM(conversions) > 0 THEN SUM(spend) / SUM(conversions) ELSE NULL END AS cost_per_conversion,
  CASE WHEN SUM(spend) > 0 THEN SUM(conversions_value) / SUM(spend) ELSE NULL END AS roas,
  SUM(engagements) AS engagements,
  SUM(video_views) AS video_views,
  COUNT(DISTINCT adset_id) AS adset_count,
  COUNT(DISTINCT ad_id) AS ad_count,
  MAX(gold_processed_at) AS gold_processed_at
FROM {SRC}
GROUP BY
  platform, full_date, year, month, month_name, day_name,
  account_id, campaign_id
''')
print("[OK] Gold.vw_campaign_performance")



In [ ]:
# Also keep platform-specific aliases for convenience
spark.sql(f"CREATE OR REPLACE VIEW {S}.vw_unified_ad_performance AS SELECT * FROM {SRC}")
spark.sql(f"CREATE OR REPLACE VIEW {S}.vw_meta_ad_performance AS SELECT * FROM {S}.rpt_meta_ad_performance_daily")
spark.sql(f"CREATE OR REPLACE VIEW {S}.vw_google_ad_performance AS SELECT * FROM {S}.rpt_google_ad_performance_daily")

print("=== VALIDATE ===")
for v in ["vw_ad_performance", "vw_adset_performance", "vw_campaign_performance"]:
    df = spark.table(f"{S}.{v}")
    print(v, "rows", df.count())
    df.groupBy("platform").agg(
        __import__("pyspark").sql.functions.count("*").alias("rows"),
        __import__("pyspark").sql.functions.round(__import__("pyspark").sql.functions.sum("spend"), 2).alias("spend"),
    ).show()

from pyspark.sql import functions as F
print("campaign sample")
spark.table(f"{S}.vw_campaign_performance").select(
    "platform","account_name","campaign_name","full_date","spend","clicks","leads","adset_count","ad_count"
).orderBy(F.col("spend").desc()).show(5, truncate=40)
print("adset sample")
spark.table(f"{S}.vw_adset_performance").select(
    "platform","campaign_name","adset_name","full_date","spend","clicks","leads","age_range","geo_cities","ad_count"
).orderBy(F.col("spend").desc()).show(5, truncate=40)
print("ad sample")
spark.table(f"{S}.vw_ad_performance").select(
    "platform","campaign_name","adset_name","ad_name","full_date","spend","clicks","leads"
).orderBy(F.col("spend").desc()).show(5, truncate=40)
print("LEVEL_VIEWS_COMPLETE")

